In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

SEARCH_DIRS = [
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "audit",
]

REQUIRED_GROUPS = {
    "implied_variance": [
        "implied_variance",
        "vix_style_vol",
        "tenor",
        "trade_date",
    ],
    "forecast_denominator": [
        "forecast_variance_candidate",
        "predicted_log_variance_candidate",
        "model_vrp_log",
        "tenor",
        "trade_date",
    ],
    "z_scores": [
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "z_3m",
        "z_1y",
        "trade_date",
        "tenor",
    ],
    "rsi": [
        "rsi14",
        "rsi_14",
        "trade_date",
    ],
    "rv21d": [
        "rv21d_vol_pct",
        "spy_close",
        "trade_date",
    ],
    "sizing_lock": [
        "selected_size_pct",
        "locked_size",
        "sleeve",
        "layer",
        "bucket",
        "dte",
    ],
    "selection_rule": [
        "selection_rule",
        "final_one_trade_per_day",
        "naked_put",
    ],
}

def get_columns(path: Path):
    try:
        if path.suffix.lower() == ".parquet":
            return list(pd.read_parquet(path, engine="pyarrow").columns)
        if path.suffix.lower() == ".csv":
            return list(pd.read_csv(path, nrows=5).columns)
        return []
    except Exception as e:
        return [f"ERROR: {type(e).__name__}: {e}"]

def score_file(cols, terms):
    cols_lower = [str(c).lower() for c in cols]
    score = 0
    hits = []
    for term in terms:
        term_lower = term.lower()
        matched = any(term_lower in c for c in cols_lower)
        if matched:
            score += 1
            hits.append(term)
    return score, hits

all_files = []

for search_dir in SEARCH_DIRS:
    if not search_dir.exists():
        continue

    for path in search_dir.rglob("*"):
        if path.suffix.lower() not in [".parquet", ".csv", ".json"]:
            continue

        all_files.append(path)

print("=" * 100)
print("Files scanned")
print("=" * 100)
print(f"Total files: {len(all_files)}")

results = []

for path in all_files:
    suffix = path.suffix.lower()

    if suffix == ".json":
        cols = []
        text = ""
        try:
            text = path.read_text(encoding="utf-8", errors="ignore")[:5000].lower()
        except Exception:
            text = ""

        for group, terms in REQUIRED_GROUPS.items():
            hits = [t for t in terms if t.lower() in text]
            if hits:
                results.append(
                    {
                        "group": group,
                        "score": len(hits),
                        "hits": ", ".join(hits),
                        "rows": None,
                        "cols": None,
                        "path": str(path),
                    }
                )
        continue

    cols = get_columns(path)

    rows = None
    try:
        if suffix == ".parquet":
            rows = len(pd.read_parquet(path, columns=[]))
        elif suffix == ".csv":
            rows = sum(1 for _ in open(path, "r", encoding="utf-8", errors="ignore")) - 1
    except Exception:
        rows = None

    for group, terms in REQUIRED_GROUPS.items():
        score, hits = score_file(cols, terms)
        if score > 0:
            results.append(
                {
                    "group": group,
                    "score": score,
                    "hits": ", ".join(hits),
                    "rows": rows,
                    "cols": len(cols),
                    "path": str(path),
                }
            )

results_df = pd.DataFrame(results)

if results_df.empty:
    print("No candidate files found.")
else:
    results_df = results_df.sort_values(["group", "score", "path"], ascending=[True, False, True])

    for group in sorted(results_df["group"].unique()):
        print("\n" + "=" * 100)
        print(group)
        print("=" * 100)

        display(
            results_df[results_df["group"] == group]
            .head(25)
            .reset_index(drop=True)
        )

Files scanned
Total files: 2190

forecast_denominator


,group,score,hits,rows,cols,path
0,forecast_denominator,5,"forecast_variance_candidate, predicted_log_var...",3112.0,129.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
1,forecast_denominator,5,"forecast_variance_candidate, predicted_log_var...",7502.0,124.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
2,forecast_denominator,5,"forecast_variance_candidate, predicted_log_var...",1029.0,127.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
3,forecast_denominator,5,"forecast_variance_candidate, predicted_log_var...",2663.0,129.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
4,forecast_denominator,5,"forecast_variance_candidate, predicted_log_var...",2663.0,150.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
5,forecast_denominator,5,"forecast_variance_candidate, predicted_log_var...",365.0,156.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
6,forecast_denominator,5,"forecast_variance_candidate, predicted_log_var...",20838.0,142.0,C:\Users\patri\vrp_project\data\audit\vrp_unif...
7,forecast_denominator,5,"forecast_variance_candidate, predicted_log_var...",20838.0,142.0,C:\Users\patri\vrp_project\data\audit\vrp_unif...
8,forecast_denominator,5,"forecast_variance_candidate, predicted_log_var...",150.0,147.0,C:\Users\patri\vrp_project\data\audit\vrp_unif...
9,forecast_denominator,5,"forecast_variance_candidate, predicted_log_var...",150.0,147.0,C:\Users\patri\vrp_project\data\audit\vrp_unif...



implied_variance


,group,score,hits,rows,cols,path
0,implied_variance,4,"implied_variance, vix_style_vol, tenor, trade_...",1000.0,65.0,C:\Users\patri\vrp_project\data\audit\forecast...
1,implied_variance,4,"implied_variance, vix_style_vol, tenor, trade_...",25.0,56.0,C:\Users\patri\vrp_project\data\audit\forecast...
2,implied_variance,4,"implied_variance, vix_style_vol, tenor, trade_...",50.0,46.0,C:\Users\patri\vrp_project\data\audit\forecast...
3,implied_variance,4,"implied_variance, vix_style_vol, tenor, trade_...",75.0,50.0,C:\Users\patri\vrp_project\data\audit\forecast...
4,implied_variance,4,"implied_variance, vix_style_vol, tenor, trade_...",75.0,61.0,C:\Users\patri\vrp_project\data\audit\forecast...
5,implied_variance,4,"implied_variance, vix_style_vol, tenor, trade_...",500.0,46.0,C:\Users\patri\vrp_project\data\audit\forecast...
6,implied_variance,4,"implied_variance, vix_style_vol, tenor, trade_...",500.0,395.0,C:\Users\patri\vrp_project\data\audit\forecast...
7,implied_variance,4,"implied_variance, vix_style_vol, tenor, trade_...",1000.0,61.0,C:\Users\patri\vrp_project\data\audit\forecast...
8,implied_variance,4,"implied_variance, vix_style_vol, tenor, trade_...",750.0,50.0,C:\Users\patri\vrp_project\data\audit\forecast...
9,implied_variance,4,"implied_variance, vix_style_vol, tenor, trade_...",1000.0,37.0,C:\Users\patri\vrp_project\data\audit\forecast...



rsi


...

,group,score,hits,rows,cols,path
0,selection_rule,3,"selection_rule, final_one_trade_per_day, naked...",NaN,NaN,C:\Users\patri\vrp_project\data\audit\vrp_core...
1,selection_rule,2,"final_one_trade_per_day, naked_put",NaN,NaN,C:\Users\patri\vrp_project\data\audit\vrp_core...
2,selection_rule,2,"final_one_trade_per_day, naked_put",NaN,NaN,C:\Users\patri\vrp_project\data\audit\vrp_core...
3,selection_rule,1,selection_rule,577.0,66.0,C:\Users\patri\vrp_project\data\audit\locked_2...
4,selection_rule,1,selection_rule,577.0,60.0,C:\Users\patri\vrp_project\data\audit\locked_2...
5,selection_rule,1,selection_rule,577.0,38.0,C:\Users\patri\vrp_project\data\audit\locked_2...
6,selection_rule,1,selection_rule,25.0,38.0,C:\Users\patri\vrp_project\data\audit\locked_2...
7,selection_rule,1,selection_rule,10.0,38.0,C:\Users\patri\vrp_project\data\audit\locked_2...
8,selection_rule,1,selection_rule,577.0,58.0,C:\Users\patri\vrp_project\data\audit\locked_2...
9,selection_rule,1,selection_rule,577.0,51.0,C:\Users\patri\vrp_project\data\audit\locked_2...



sizing_lock


,group,score,hits,rows,cols,path
0,sizing_lock,4,"sleeve, layer, bucket, dte",3112.0,129.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
1,sizing_lock,4,"sleeve, layer, bucket, dte",7502.0,124.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
2,sizing_lock,4,"sleeve, layer, bucket, dte",1029.0,127.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
3,sizing_lock,4,"sleeve, layer, bucket, dte",2663.0,129.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
4,sizing_lock,4,"sleeve, layer, bucket, dte",2663.0,150.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
5,sizing_lock,4,"sleeve, layer, bucket, dte",NaN,NaN,C:\Users\patri\vrp_project\data\audit\vrp_core...
6,sizing_lock,4,"sleeve, layer, bucket, dte",365.0,156.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
7,sizing_lock,4,"locked_size, sleeve, layer, bucket",13.0,43.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
8,sizing_lock,4,"locked_size, sleeve, layer, bucket",365.0,18.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
9,sizing_lock,4,"locked_size, sleeve, layer, bucket",13.0,35.0,C:\Users\patri\vrp_project\data\audit\vrp_core...



z_scores


,group,score,hits,rows,cols,path
0,z_scores,6,"model_vrp_z_3m, model_vrp_z_1y, z_3m, z_1y, tr...",240.0,44.0,C:\Users\patri\vrp_project\data\audit\vrp_30d_...
1,z_scores,6,"model_vrp_z_3m, model_vrp_z_1y, z_3m, z_1y, tr...",140.0,42.0,C:\Users\patri\vrp_project\data\audit\vrp_30d_...
2,z_scores,6,"model_vrp_z_3m, model_vrp_z_1y, z_3m, z_1y, tr...",40.0,44.0,C:\Users\patri\vrp_project\data\audit\vrp_30d_...
3,z_scores,6,"model_vrp_z_3m, model_vrp_z_1y, z_3m, z_1y, tr...",17568.0,9.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
4,z_scores,6,"model_vrp_z_3m, model_vrp_z_1y, z_3m, z_1y, tr...",17568.0,9.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
5,z_scores,6,"model_vrp_z_3m, model_vrp_z_1y, z_3m, z_1y, tr...",21448.0,28.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
6,z_scores,6,"model_vrp_z_3m, model_vrp_z_1y, z_3m, z_1y, tr...",13369.0,31.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
7,z_scores,6,"model_vrp_z_3m, model_vrp_z_1y, z_3m, z_1y, tr...",21428.0,31.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
8,z_scores,6,"model_vrp_z_3m, model_vrp_z_1y, z_3m, z_1y, tr...",3346.0,31.0,C:\Users\patri\vrp_project\data\audit\vrp_core...
9,z_scores,6,"model_vrp_z_3m, model_vrp_z_1y, z_3m, z_1y, tr...",1975.0,31.0,C:\Users\patri\vrp_project\data\audit\vrp_core...


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

SEARCH_ROOTS = [
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "audit",
]

KEY_PATTERNS = [
    "07A",
    "unified_fds_no_min_return",
    "oos_forecast_panel",
    "forecast_panel",
    "core_secondary_consistent_secondary",
    "final_one_trade_per_day",
    "naked_put",
    "15S_final_one_trade_per_day",
    "sizing_lock",
    "selection_rule",
    "16S_locked",
    "cumulative_pnl",
]

KEY_COLUMNS = [
    "trade_date",
    "tenor",
    "implied_variance",
    "vix_style_vol",
    "forecast_variance_candidate",
    "predicted_log_variance_candidate",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "rsi14",
    "rv21d_vol_pct",
    "selected_size_pct",
    "locked_size",
    "sleeve",
    "signal_layer",
    "signal_bucket",
    "selected_layer",
    "selected_tenor",
]

def file_matches(path: Path):
    s = str(path).lower()
    return any(p.lower() in s for p in KEY_PATTERNS)

def safe_read_cols(path: Path):
    try:
        if path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path)
        elif path.suffix.lower() == ".csv":
            df = pd.read_csv(path)
        else:
            return None, []
        return df, list(df.columns)
    except Exception as e:
        return None, [f"ERROR: {type(e).__name__}: {e}"]

rows = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if path.suffix.lower() not in [".parquet", ".csv", ".json"]:
            continue

        if not file_matches(path):
            continue

        stat = path.stat()
        mtime = datetime.fromtimestamp(stat.st_mtime)

        if path.suffix.lower() == ".json":
            text = path.read_text(encoding="utf-8", errors="ignore")[:10000].lower()
            hits = [p for p in KEY_PATTERNS if p.lower() in text or p.lower() in str(path).lower()]
            rows.append({
                "path": str(path),
                "suffix": path.suffix,
                "modified": mtime,
                "rows": None,
                "cols": None,
                "min_date": None,
                "max_date": None,
                "tenors": None,
                "key_col_hits": None,
                "pattern_hits": ", ".join(hits),
            })
            continue

        df, cols = safe_read_cols(path)

        if df is None:
            rows.append({
                "path": str(path),
                "suffix": path.suffix,
                "modified": mtime,
                "rows": None,
                "cols": None,
                "min_date": None,
                "max_date": None,
                "tenors": None,
                "key_col_hits": "; ".join(cols),
                "pattern_hits": None,
            })
            continue

        min_date = None
        max_date = None
        tenors = None

        if "trade_date" in df.columns and len(df) > 0:
            d = pd.to_datetime(df["trade_date"], errors="coerce")
            if d.notna().any():
                min_date = str(d.min().date())
                max_date = str(d.max().date())

        if "tenor" in df.columns and len(df) > 0:
            try:
                tenors = sorted(pd.Series(df["tenor"]).dropna().astype(int).unique().tolist())
            except Exception:
                tenors = sorted(pd.Series(df["tenor"]).dropna().astype(str).unique().tolist())

        key_col_hits = [c for c in KEY_COLUMNS if c in df.columns]
        pattern_hits = [p for p in KEY_PATTERNS if p.lower() in str(path).lower()]

        rows.append({
            "path": str(path),
            "suffix": path.suffix,
            "modified": mtime,
            "rows": len(df),
            "cols": len(df.columns),
            "min_date": min_date,
            "max_date": max_date,
            "tenors": tenors,
            "key_col_hits": ", ".join(key_col_hits),
            "pattern_hits": ", ".join(pattern_hits),
        })

inventory = pd.DataFrame(rows)

if inventory.empty:
    print("No targeted candidate files found.")
else:
    inventory["score"] = (
        inventory["key_col_hits"].fillna("").str.count(",") + 1
        + inventory["pattern_hits"].fillna("").str.count(",") + 1
        + inventory["rows"].fillna(0).clip(upper=10000) / 10000
    )

    inventory = inventory.sort_values(
        ["score", "modified", "rows"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    out_path = PROJECT_ROOT / "data" / "audit" / "vrp_final_signal_source_inventory_targeted.csv"
    inventory.to_csv(out_path, index=False)

    print("Saved targeted inventory:")
    print(out_path)

    display(
        inventory[
            [
                "path",
                "modified",
                "rows",
                "cols",
                "min_date",
                "max_date",
                "tenors",
                "key_col_hits",
                "pattern_hits",
                "score",
            ]
        ].head(100)
    )

Saved targeted inventory:
C:\Users\patri\vrp_project\data\audit\vrp_final_signal_source_inventory_targeted.csv


,path,modified,rows,cols,min_date,max_date,tenors,key_col_hits,pattern_hits,score
0,C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\07A_unified_fds_no_min_return_oos_forecast_panel_20200102_20260701_20260704_203242.parquet,2026-07-04 20:32:49.275804,31140.0,89.0,2019-01-02,2026-07-01,"[9, 12, 15, 18, 21, 24, 27, 30, 33]","trade_date, tenor, implied_variance, vix_style_vol, forecast_variance_candidate, predicted_log_variance_candidate, model_vrp_log, model_vrp_z_3m, model_vrp_z_1y, RSI14","07A, unified_fds_no_min_return, oos_forecast_panel, forecast_panel",15.0000
1,C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\06C_back_model_spec_oos_forecast_panel_20200102_20260701_20260704_202315.parquet,2026-07-04 20:23:37.614007,49212.0,93.0,2019-01-02,2026-07-01,"[27, 30, 33]","trade_date, tenor, implied_variance, vix_style_vol, forecast_variance_candidate, predicted_log_variance_candidate, model_vrp_log, model_vrp_z_3m, model_vrp_z_1y, RSI14","oos_forecast_panel, forecast_panel",13.0000
2,C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\06B_back_unfreeze_oos_forecast_panel_20200102_20260701_20260704_200958.parquet,2026-07-04 20:10:06.902388,24732.0,89.0,2019-01-02,2026-07-01,"[27, 30, 33]","trade_date, tenor, implied_variance, vix_style_vol, forecast_variance_candidate, predicted_log_variance_candidate, model_vrp_log, model_vrp_z_3m, model_vrp_z_1y, RSI14","oos_forecast_panel, forecast_panel",13.0000
3,C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\07A_unified_signal_threshold_diagnostic_panel_20200102_20260701_20260704_203242.parquet,2026-07-04 20:32:49.632170,29376.0,99.0,2020-01-02,2026-07-01,"[9, 12, 15, 18, 21, 24, 27, 30, 33]","trade_date, tenor, implied_variance, vix_style_vol, forecast_variance_candidate, predicted_log_variance_candidate, model_vrp_log, model_vrp_z_3m, model_vrp_z_1y, RSI14",07A,12.0000
4,C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\07A_unified_common_row_panel_20200102_20260701_20260704_203242.parquet,2026-07-04 20:32:49.448847,29376.0,93.0,2020-01-02,2026-07-01,"[9, 12, 15, 18, 21, 24, 27, 30, 33]","trade_date, tenor, implied_variance, vix_style_vol, forecast_variance_candidate, predicted_log_variance_candidate, model_vrp_log, model_vrp_z_3m, model_vrp_z_1y, RSI14",07A,12.0000
5,C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\07A_unified_qualified_trade_diagnostic_panel_20200102_20260701_20260704_203242.parquet,2026-07-04 20:32:49.668527,3361.0,99.0,2020-01-03,2026-05-04,"[9, 12, 15, 18, 21, 24, 27, 30, 33]","trade_date, tenor, implied_variance, vix_style_vol, forecast_variance_candidate, predicted_log_variance_candidate, model_vrp_log, model_vrp_z_3m, model_vrp_z_1y, RSI14",07A,11.3361
6,C:\Users\patri\vrp_project\data\audit\vrp_core_secondary_tertiary_independent_sizing_research_v1\16S_locked_dte_performance_manifest_20260706_143802.json,2026-07-06 14:38:02.754414,NaN,NaN,None,None,None,None,"final_one_trade_per_day, naked_put, 15S_final_one_trade_per_day, sizing_lock, 16S_locked, cumulative_pnl",7.0000
7,C:\Users\patri\vrp_project\data\audit\vrp_core_secondary_tertiary_independent_sizing_research_v1\15S_final_one_trade_per_day_naked_put_sizing_lock_manifest_20260706_142800.json,2026-07-06 14:28:00.582081,NaN,NaN,None,None,None,None,"core_secondary_consistent_secondary, final_one_trade_per_day, naked_put, 15S_final_one_trade_per_day, sizing_lock, selection_rule",7.0000
8,C:\Users\patri\vrp_project\data\processed\vrp_core_secondary_tertiary_independent_sizing_research_v1\15S_final_one_trade_per_day_naked_put_sizing_lock_rules_20260706_142800.parquet,2026-07-06 14:28:00.540486,13.0,43.0,None,None,"[12, 15, 18, 21, 24, 27, 30, 33]","tenor, signal_layer","final_one_trade_per_day, naked_put, 15S_final_one_trade_per_day, sizing_lock",6.0013
9,C:\Users\patri\vrp_project\data\audit\vrp_core_secondary

In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

FORECAST_PANEL_PATH = Path(
    r"C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1"
    r"\07A_unified_fds_no_min_return_oos_forecast_panel_20200102_20260701_20260704_203242.parquet"
)

SIGNAL_RULES_PATH = Path(
    r"C:\Users\patri\vrp_project\data\processed\vrp_core_secondary_tertiary_independent_sizing_research_v1"
    r"\09S_core_secondary_consistent_secondary_locked_rules_long_20260705_221829.parquet"
)

SIZING_RULES_PATH = Path(
    r"C:\Users\patri\vrp_project\data\processed\vrp_core_secondary_tertiary_independent_sizing_research_v1"
    r"\15S_final_one_trade_per_day_naked_put_sizing_lock_rules_20260706_142800.parquet"
)

SELECTION_RULE_PATH = Path(
    r"C:\Users\patri\vrp_project\data\audit\vrp_core_secondary_tertiary_independent_sizing_research_v1"
    r"\15S_final_one_trade_per_day_naked_put_selection_rule_20260706_142800.json"
)

RV21D_PATH = Path(
    r"C:\Users\patri\vrp_project\data\processed\market_data\spy_realized_vol_history_v1.parquet"
)

paths = {
    "forecast_panel": FORECAST_PANEL_PATH,
    "signal_rules": SIGNAL_RULES_PATH,
    "sizing_rules": SIZING_RULES_PATH,
    "selection_rule": SELECTION_RULE_PATH,
    "rv21d": RV21D_PATH,
}

print("=" * 100)
print("Source file existence check")
print("=" * 100)

for name, path in paths.items():
    print(f"{name:20s} exists={path.exists()}  path={path}")

missing = [name for name, path in paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing source files: {missing}")

# ======================================================================================
# Load sources
# ======================================================================================

forecast = pd.read_parquet(FORECAST_PANEL_PATH)
signal_rules = pd.read_parquet(SIGNAL_RULES_PATH)
sizing_rules = pd.read_parquet(SIZING_RULES_PATH)
rv21d = pd.read_parquet(RV21D_PATH)

with open(SELECTION_RULE_PATH, "r", encoding="utf-8") as f:
    selection_rule = json.load(f)

# Normalize dates for inspection
for df_name, df in [
    ("forecast", forecast),
    ("rv21d", rv21d),
]:
    if "trade_date" in df.columns:
        df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce")

# ======================================================================================
# Basic summaries
# ======================================================================================

def summarize_panel(name, df):
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)
    print(f"Rows: {len(df):,}")
    print(f"Cols: {len(df.columns):,}")
    
    if "trade_date" in df.columns:
        d = pd.to_datetime(df["trade_date"], errors="coerce")
        print(f"Date range: {d.min().date()} to {d.max().date()}")
        print(f"Unique dates: {d.nunique():,}")
    
    if "tenor" in df.columns:
        try:
            tenors = sorted(pd.Series(df["tenor"]).dropna().astype(int).unique().tolist())
        except Exception:
            tenors = sorted(pd.Series(df["tenor"]).dropna().astype(str).unique().tolist())
        print(f"Tenors: {tenors}")
        print(f"Unique tenors: {len(tenors)}")
    
    print("\nColumns:")
    for c in df.columns:
        print(f"  {c}")

summarize_panel("FORECAST PANEL — 07A unified FDS no-min-return OOS forecast panel", forecast)
summarize_panel("SIGNAL RULES — 09S Core/Secondary locked rules long", signal_rules)
summarize_panel("SIZING RULES — 15S final naked-put sizing lock rules", sizing_rules)
summarize_panel("RV21D — canonical SPY realized-vol history", rv21d)

print("\n" + "=" * 100)
print("SELECTION RULE JSON")
print("=" * 100)
print(json.dumps(selection_rule, indent=2)[:5000])

# ======================================================================================
# Forecast panel contract checks
# ======================================================================================

print("\n" + "=" * 100)
print("Forecast panel contract checks")
print("=" * 100)

required_forecast_cols = [
    "trade_date",
    "tenor",
    "implied_variance",
    "vix_style_vol",
    "forecast_variance_candidate",
    "predicted_log_variance_candidate",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
]

forecast_col_check = pd.DataFrame({
    "column": required_forecast_cols,
    "exists": [c in forecast.columns for c in required_forecast_cols],
})

display(forecast_col_check)

if not forecast_col_check["exists"].all():
    missing = forecast_col_check.loc[~forecast_col_check["exists"], "column"].tolist()
    raise RuntimeError(f"Forecast panel missing required columns: {missing}")

# One row per trade_date x tenor?
dup_mask = forecast.duplicated(["trade_date", "tenor"], keep=False)
dup_count = int(dup_mask.sum())

tenor_counts = (
    forecast.groupby("trade_date")["tenor"]
    .nunique()
    .reset_index(name="unique_tenors")
)

print(f"Duplicate trade_date x tenor rows: {dup_count:,}")
print("\nTenor count distribution by date:")
display(tenor_counts["unique_tenors"].value_counts().sort_index().reset_index().rename(
    columns={"index": "unique_tenors", "unique_tenors": "date_count"}
))

if dup_count > 0:
    print("\nSample duplicate rows:")
    display(
        forecast.loc[
            dup_mask,
            [
                "trade_date",
                "tenor",
                "implied_variance",
                "forecast_variance_candidate",
                "model_vrp_log",
                "model_vrp_z_3m",
                "model_vrp_z_1y",
                "RSI14",
            ],
        ]
        .sort_values(["trade_date", "tenor"])
        .head(50)
    )

# Expected tenor grid
expected_tenors = [9, 12, 15, 18, 21, 24, 27, 30, 33]

actual_tenors = sorted(forecast["tenor"].dropna().astype(int).unique().tolist())
print(f"\nExpected tenors: {expected_tenors}")
print(f"Actual tenors:   {actual_tenors}")

# Reconstruct forecast vol and model_vrp_log
calc = forecast.copy()

calc["forecast_vol_recalc"] = np.sqrt(calc["forecast_variance_candidate"]) * 100.0
calc["model_vrp_log_recalc"] = np.log(calc["implied_variance"] / calc["forecast_variance_candidate"])

calc["forecast_vol_recalc_finite"] = np.isfinite(calc["forecast_vol_recalc"])
calc["model_vrp_log_diff"] = calc["model_vrp_log"] - calc["model_vrp_log_recalc"]

print("\nForecast variance checks:")
print("forecast_variance_candidate positive:",
      bool((calc["forecast_variance_candidate"] > 0).all()))
print("implied_variance positive:",
      bool((calc["implied_variance"] > 0).all()))
print("forecast_vol_recalc finite:",
      bool(calc["forecast_vol_recalc_finite"].all()))

print("\nmodel_vrp_log reconstruction error:")
display(
    calc["model_vrp_log_diff"]
    .abs()
    .describe(percentiles=[0.5, 0.9, 0.99, 0.999])
    .to_frame("abs_diff")
)

bad_vrp = calc[calc["model_vrp_log_diff"].abs() > 1e-10]

print(f"Rows where model_vrp_log differs from log(implied/forecast) by > 1e-10: {len(bad_vrp):,}")

if len(bad_vrp) > 0:
    display(
        bad_vrp[
            [
                "trade_date",
                "tenor",
                "implied_variance",
                "forecast_variance_candidate",
                "model_vrp_log",
                "model_vrp_log_recalc",
                "model_vrp_log_diff",
            ]
        ].head(50)
    )

# ======================================================================================
# RV21D join inspection
# ======================================================================================

print("\n" + "=" * 100)
print("RV21D join inspection")
print("=" * 100)

required_rv_cols = ["trade_date", "spy_close", "rv21d_vol_pct"]

rv_col_check = pd.DataFrame({
    "column": required_rv_cols,
    "exists": [c in rv21d.columns for c in required_rv_cols],
})

display(rv_col_check)

if not rv_col_check["exists"].all():
    missing = rv_col_check.loc[~rv_col_check["exists"], "column"].tolist()
    raise RuntimeError(f"RV21D file missing required columns: {missing}")

rv_join = forecast[["trade_date", "tenor"]].drop_duplicates().merge(
    rv21d[["trade_date", "spy_close", "rv21d_vol_pct"]],
    on="trade_date",
    how="left",
)

missing_rv = rv_join["rv21d_vol_pct"].isna().sum()

print(f"Forecast date-tenor rows: {len(rv_join):,}")
print(f"Missing rv21d after join: {missing_rv:,}")

if missing_rv > 0:
    print("\nMissing RV21D sample:")
    display(rv_join[rv_join["rv21d_vol_pct"].isna()].head(50))

print("\nLatest joined RV21D rows:")
display(
    rv_join.sort_values(["trade_date", "tenor"])
    .tail(18)
)

# ======================================================================================
# Signal rules and sizing rules inspection
# ======================================================================================

print("\n" + "=" * 100)
print("Signal rules inspection")
print("=" * 100)

display(signal_rules.head(50))

print("\nSignal rules columns:")
for c in signal_rules.columns:
    print(f"  {c}")

print("\n" + "=" * 100)
print("Sizing rules inspection")
print("=" * 100)

display(sizing_rules.head(50))

print("\nSizing rules columns:")
for c in sizing_rules.columns:
    print(f"  {c}")

# Guess likely sizing columns
possible_size_cols = [
    c for c in sizing_rules.columns
    if any(x in c.lower() for x in ["size", "sizing", "locked"])
]

print("\nPossible sizing columns:")
print(possible_size_cols)

# ======================================================================================
# Suggested source contract summary
# ======================================================================================

print("\n" + "=" * 100)
print("Suggested final source contract summary")
print("=" * 100)

summary = {
    "forecast_panel_path": str(FORECAST_PANEL_PATH),
    "forecast_rows": int(len(forecast)),
    "forecast_min_date": str(forecast["trade_date"].min().date()),
    "forecast_max_date": str(forecast["trade_date"].max().date()),
    "forecast_tenors": actual_tenors,
    "forecast_duplicate_trade_date_tenor_rows": dup_count,
    "rv21d_path": str(RV21D_PATH),
    "rv21d_rows": int(len(rv21d)),
    "rv21d_min_date": str(rv21d["trade_date"].min().date()),
    "rv21d_max_date": str(rv21d["trade_date"].max().date()),
    "missing_rv21d_after_join": int(missing_rv),
    "signal_rules_path": str(SIGNAL_RULES_PATH),
    "signal_rules_rows": int(len(signal_rules)),
    "sizing_rules_path": str(SIZING_RULES_PATH),
    "sizing_rules_rows": int(len(sizing_rules)),
    "selection_rule_path": str(SELECTION_RULE_PATH),
}

print(json.dumps(summary, indent=2))

Source file existence check
forecast_panel       exists=True  path=C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\07A_unified_fds_no_min_return_oos_forecast_panel_20200102_20260701_20260704_203242.parquet
signal_rules         exists=True  path=C:\Users\patri\vrp_project\data\processed\vrp_core_secondary_tertiary_independent_sizing_research_v1\09S_core_secondary_consistent_secondary_locked_rules_long_20260705_221829.parquet
sizing_rules         exists=True  path=C:\Users\patri\vrp_project\data\processed\vrp_core_secondary_tertiary_independent_sizing_research_v1\15S_final_one_trade_per_day_naked_put_sizing_lock_rules_20260706_142800.parquet
selection_rule       exists=True  path=C:\Users\patri\vrp_project\data\audit\vrp_core_secondary_tertiary_independent_sizing_research_v1\15S_final_one_trade_per_day_naked_put_selection_rule_20260706_142800.json
rv21d                exists=True  path=C:\Users\patri\vrp_project\data\processed\market_data\spy_realized_

,column,exists
0,trade_date,True
1,tenor,True
2,implied_variance,True
3,vix_style_vol,True
4,forecast_variance_candidate,True
5,predicted_log_variance_candidate,True
6,model_vrp_log,True
7,model_vrp_z_3m,True
8,model_vrp_z_1y,True
9,RSI14,True


Duplicate trade_date x tenor rows: 29,376

Tenor count distribution by date:


,date_count,count
0,7,252
1,9,1632



Sample duplicate rows:


...


Signal rules inspection


,stack_lock_version,stack_lock_decision_id,model_spec,signal_layer,signal_priority,lock_version,lock_decision_id,supersedes_lock_version,supersedes_lock_decision_id,tenor,tenor_bucket,tenor_bucket_order,include_tenor,model_vrp_log_min,model_vrp_z_3m_min,model_vrp_z_1y_min,RSI14_max,rv21d_vol_pct_min,comparison_operator_model_vrp_log,comparison_operator_model_vrp_z_3m,comparison_operator_model_vrp_z_1y,comparison_operator_RSI14,comparison_operator_rv21d_vol_pct,vrp_measure,forecast_model,rv_floor_contract,selection_rule,source_cell,source_candidate_id,exclusion_reason,created_timestamp,supersedes_stack_lock_version,supersedes_stack_lock_decision_id
0,core_secondary_consistent_secondary_signal_stack_locked_v1,core_secondary_consistent_secondary_signal_stack_locked_001,unified_fds_no_min_return,Core,1,core_middle_back_only_locked_v1,core_middle_back_only_locked_001,core_repaired_v1,core_repaired_v1_locked_001,12,Front,1,False,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,model_vrp_log = log(implied_variance / forecast_variance),har_total_simple / unified_fds_no_min_return,rv21d_vol_pct > threshold,"Core Middle/Back first. If any locked Core Middle/Back tenor qualifies on a date, select Core and ignore Secondary. If no Core tenor qualifies, evaluate Secondary Front/Middle/Back. If multiple Secondary tenors qualify, select by BEST_SIGNAL_RANK. No active Tertiary layer.",Cell 7S — Front taxonomy architecture review,C_revised_front_as_secondary_stack,Core Front removed after Cell 7S architecture review. Front exposure is handled by Secondary Front only.,20260705_221829,core_secondary_front_taxonomy_signal_stack_locked_v1,core_secondary_front_taxonomy_signal_stack_locked_001
1,core_secondary_consistent_secondary_signal_stack_locked_v1,core_secondary_consistent_secondary_signal_stack_locked_001,unified_fds_no_min_return,Core,1,core_middle_back_only_locked_v1,core_middle_back_only_locked_001,core_repaired_v1,core_repaired_v1_locked_001,15,Front,1,False,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,model_vrp_log = log(implied_variance / forecast_variance),har_total_simple / unified_fds_no_min_return,rv21d_vol_pct > threshold,"Core Middle/Back first. If any locked Core Middle/Back tenor qualifies on a date, select Core and ignore Secondary. If no Core tenor qualifies, evaluate Secondary Front/Middle/Back. If multiple Secondary tenors qualify, select by BEST_SIGNAL_RANK. No active Tertiary layer.",Cell 7S — Front taxonomy architecture review,C_revised_front_as_secondary_stack,Core Front removed after Cell 7S architecture review. Front exposure is handled by Secondary Front only.,20260705_221829,core_secondary_front_taxonomy_signal_stack_locked_v1,core_secondary_front_taxonomy_signal_stack_locked_001
2,core_secondary_consistent_secondary_signal_stack_locked_v1,core_secondary_consistent_secondary_signal_stack_locked_001,unified_fds_no_min_return,Core,1,core_middle_back_only_locked_v1,core_middle_back_only_locked_001,core_repaired_v1,core_repaired_v1_locked_001,18,Front,1,False,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,model_vrp_log = log(implied_variance / forecast_variance),har_total_simple / unified_fds_no_min_return,rv21d_vol_pct > threshold,"Core Middle/Back first. If any locked Core Middle/Back tenor qualifies on a date, select Core and ignore Secondary. If no Core tenor qualifies, evaluate Secondary Front/Middle/Back. If multiple Secondary tenors qualify, select by BEST_SIGNAL_RANK. No active Tertiary layer.",Cell 7S — Front taxonomy architecture review,C_revised_front_as_secondary_stack,Core Front removed after Cell 7S architecture review. Front exposure is handled by Secondary Front only.,20260705_221829,core_secondary_front_taxonomy_signal_stack_locked_v1,core_secondary_front_taxonomy_signal_stack_locked_001
3,core_secondary_consistent_secondary_signal_stack_locked_v1,core_secondary_consistent_secondary_signal_stack_locked_001,unified_fds_no_min_return,Core,1,core_middle_back_only_locked_v1,core_middle_back_only_locked_001,core_rep


Signal rules columns:
  stack_lock_version
  stack_lock_decision_id
  model_spec
  signal_layer
  signal_priority
  lock_version
  lock_decision_id
  supersedes_lock_version
  supersedes_lock_decision_id
  tenor
  tenor_bucket
  tenor_bucket_order
  include_tenor
  model_vrp_log_min
  model_vrp_z_3m_min
  model_vrp_z_1y_min
  RSI14_max
  rv21d_vol_pct_min
  comparison_operator_model_vrp_log
  comparison_operator_model_vrp_z_3m
  comparison_operator_model_vrp_z_1y
  comparison_operator_RSI14
  comparison_operator_rv21d_vol_pct
  vrp_measure
  forecast_model
  rv_floor_contract
  selection_rule
  source_cell
  source_candidate_id
  exclusion_reason
  created_timestamp
  supersedes_stack_lock_version
  supersedes_stack_lock_decision_id

Sizing rules inspection


,sleeve_id,locked_size_pct,signal_layer,tenor_bucket,tenor,model_size_pct_label,manual_size_pct_label,manual_minus_model_size_pct,tail_budget_rounded_size_pct,final_recommended_size_pct,sizing_quality_score,win_rate,avg_pnl_per_day,profit_factor_pnl_per_day,pnl_per_day_expected_loss_1pct_positive,worst20_loss_positive,avg_pnl_per_day_2026,primary_size_bottleneck,locked_size_label,parsed_signal_layer,parsed_tenor_bucket,parsed_tenor,sizing_lock_version,sizing_lock_decision_id,program_type,program_label,signal_stack_lock_version,signal_stack_lock_decision_id,is_locked,is_final_sizing_lock,locked_at,max_trades_per_day,selection_primary_key,tie_break_1,tie_break_2,tie_break_3,tie_break_4,tie_break_5,pricing_basis_label,not_defined_spread_sizing,signal_layer_order,tenor_bucket_order,display_order
0,Core_Middle_21D,0.0350,Core,Middle,21,3.50%,3.50%,0.0000,0.0350,0.0350,0.791050,0.849315,582.804165,3.813833,3935.354296,12713.840942,-63.750780,worst_trade,3.50%,Core,Middle,21,final_one_trade_per_day_naked_put_sizing_locked_v1,final_one_trade_per_day_naked_put_sizing_locked_001,naked_short_put_tenor_selection_and_sizing,Final one-trade-per-day naked short put sizing program,core_secondary_consistent_secondary_signal_stack_locked_v1,core_secondary_consistent_secondary_signal_stack_locked_001,True,True,20260706_142800,1,highest_locked_size_pct,Core beats Secondary,higher sizing_quality_score,higher sleeve win_rate,lower 1pct_expected_loss,longer DTE,short_put_entry_credit_points,True,1,2,1
1,Core_Middle_24D,0.0425,Core,Middle,24,4.25%,4.25%,0.0000,0.0425,0.0425,0.789258,0.851351,522.835118,3.814531,3122.894260,10768.622387,-4.426712,quality_cap,4.25%,Core,Middle,24,final_one_trade_per_day_naked_put_sizing_locked_v1,final_one_trade_per_day_naked_put_sizing_locked_001,naked_short_put_tenor_selection_and_sizing,Final one-trade-per-day naked short put sizing program,core_secondary_consistent_secondary_signal_stack_locked_v1,core_secondary_consistent_secondary_signal_stack_locked_001,True,True,20260706_142800,1,highest_locked_size_pct,Core beats Secondary,higher sizing_quality_score,higher sleeve win_rate,lower 1pct_expected_loss,longer DTE,short_put_entry_credit_points,True,1,2,2
2,Core_Back_27D,0.0450,Core,Back,27,4.50%,4.50%,0.0000,0.0450,0.0450,0.850288,0.875000,567.163740,4.335512,3429.035282,5706.090687,154.602927,quality_cap,4.50%,Core,Back,27,final_one_trade_per_day_naked_put_sizing_locked_v1,final_one_trade_per_day_naked_put_sizing_locked_001,naked_short_put_tenor_selection_and_sizing,Final one-trade-per-day naked short put sizing program,core_secondary_consistent_secondary_signal_stack_locked_v1,core_secondary_consistent_secondary_signal_stack_locked_001,True,True,20260706_142800,1,highest_locked_size_pct,Core beats Secondary,higher sizing_quality_score,higher sleeve win_rate,lower 1pct_expected_loss,longer DTE,short_put_entry_credit_points,True,1,3,3
3,Core_Back_30D,0.0475,Core,Back,30,5.00%,4.75%,-0.0025,0.0500,0.0475,0.924325,0.902985,600.006302,5.756158,3270.987604,1174.786905,255.213286,max_size_cap,4.75%,Core,Back,30,final_one_trade_per_day_naked_put_sizing_locked_v1,final_one_trade_per_day_naked_put_sizing_locked_001,naked_short_put_tenor_selection_and_sizing,Final one-trade-per-day naked short put sizing program,core_secondary_consistent_secondary_signal_stack_locked_v1,core_secondary_consistent_secondary_signal_stack_locked_001,True,True,20260706_142800,1,highest_locked_size_pct,Core beats Secondary,higher sizing_quality_score,higher sleeve win_rate,lower 1pct_expected_loss,longer DTE,short_put_entry_credit_points,True,1,3,4
4,Core_Back_33D,0.0500,Core,Back,33,5.00%,5.00%,0.0000,0.0500,0.0500,0.942536,0.926471,610.808698,8.306366,2478.940162,0.000000,358.706507,max_size_cap,5.00%,Core,Back,33,final_one_trade_per_day_naked_put_sizing_locked_v1,final_one_trade_per_day_naked_put_sizing_locked_001,naked_short_put_tenor_selection_and_sizing,Final one-trade-per-day naked short put sizing program,core_secondary_consistent_secondar


Sizing rules columns:
  sleeve_id
  locked_size_pct
  signal_layer
  tenor_bucket
  tenor
  model_size_pct_label
  manual_size_pct_label
  manual_minus_model_size_pct
  tail_budget_rounded_size_pct
  final_recommended_size_pct
  sizing_quality_score
  win_rate
  avg_pnl_per_day
  profit_factor_pnl_per_day
  pnl_per_day_expected_loss_1pct_positive
  worst20_loss_positive
  avg_pnl_per_day_2026
  primary_size_bottleneck
  locked_size_label
  parsed_signal_layer
  parsed_tenor_bucket
  parsed_tenor
  sizing_lock_version
  sizing_lock_decision_id
  program_type
  program_label
  signal_stack_lock_version
  signal_stack_lock_decision_id
  is_locked
  is_final_sizing_lock
  locked_at
  max_trades_per_day
  selection_primary_key
  tie_break_1
  tie_break_2
  tie_break_3
  tie_break_4
  tie_break_5
  pricing_basis_label
  not_defined_spread_sizing
  signal_layer_order
  tenor_bucket_order
  display_order

Possible sizing columns:
['locked_size_pct', 'model_size_pct_label', 'manual_size_pct_la

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

FORECAST_PANEL_PATH = Path(
    r"C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1"
    r"\07A_unified_fds_no_min_return_oos_forecast_panel_20200102_20260701_20260704_203242.parquet"
)

COMMON_ROW_PANEL_PATH = Path(
    r"C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1"
    r"\07A_unified_common_row_panel_20200102_20260701_20260704_203242.parquet"
)

DIAGNOSTIC_PANEL_PATH = Path(
    r"C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1"
    r"\07A_unified_signal_threshold_diagnostic_panel_20200102_20260701_20260704_203242.parquet"
)

paths = {
    "forecast_panel": FORECAST_PANEL_PATH,
    "common_row_panel": COMMON_ROW_PANEL_PATH,
    "diagnostic_panel": DIAGNOSTIC_PANEL_PATH,
}

print("=" * 100)
print("07A duplicate-block diagnostic — source existence")
print("=" * 100)

for name, path in paths.items():
    print(f"{name:20s} exists={path.exists()}  path={path}")

missing = [name for name, path in paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing source files: {missing}")

# ======================================================================================
# Load panels
# ======================================================================================

forecast = pd.read_parquet(FORECAST_PANEL_PATH)
common = pd.read_parquet(COMMON_ROW_PANEL_PATH)
diagnostic = pd.read_parquet(DIAGNOSTIC_PANEL_PATH)

for df in [forecast, common, diagnostic]:
    if "trade_date" in df.columns:
        df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce")
    if "tenor" in df.columns:
        df["tenor"] = pd.to_numeric(df["tenor"], errors="coerce").astype("Int64")

# ======================================================================================
# Helper functions
# ======================================================================================

def panel_summary(name, df):
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)
    print(f"Rows: {len(df):,}")
    print(f"Cols: {len(df.columns):,}")
    if "trade_date" in df.columns:
        print(f"Date range: {df['trade_date'].min().date()} to {df['trade_date'].max().date()}")
        print(f"Unique dates: {df['trade_date'].nunique():,}")
    if "tenor" in df.columns:
        print(f"Tenors: {sorted(df['tenor'].dropna().astype(int).unique().tolist())}")
    if {"trade_date", "tenor"}.issubset(df.columns):
        dup_rows = int(df.duplicated(["trade_date", "tenor"], keep=False).sum())
        unique_keys = int(df[["trade_date", "tenor"]].drop_duplicates().shape[0])
        print(f"Unique trade_date × tenor keys: {unique_keys:,}")
        print(f"Duplicate trade_date × tenor rows: {dup_rows:,}")
        tenor_counts = (
            df.groupby("trade_date")["tenor"]
            .nunique()
            .value_counts()
            .sort_index()
            .reset_index()
        )
        tenor_counts.columns = ["unique_tenors_per_date", "date_count"]
        print("\nTenor-count distribution:")
        display(tenor_counts)

def add_reconstruction_checks(df, label):
    out = df.copy()

    var_cols = [
        "forecast_variance_candidate",
        "forecast_variance_har_downside_v1",
        "baseline_forecast_variance",
    ]

    for col in var_cols:
        if col in out.columns:
            out[f"{label}_vrp_log_using_{col}"] = np.log(out["implied_variance"] / out[col])
            out[f"{label}_abs_diff_using_{col}"] = (
                out["model_vrp_log"] - out[f"{label}_vrp_log_using_{col}"]
            ).abs()

    if "candidate_forecast_vol_pct" in out.columns and "forecast_variance_candidate" in out.columns:
        out[f"{label}_candidate_vol_recalc"] = np.sqrt(out["forecast_variance_candidate"]) * 100
        out[f"{label}_candidate_vol_abs_diff"] = (
            out["candidate_forecast_vol_pct"] - out[f"{label}_candidate_vol_recalc"]
        ).abs()

    return out

def summarize_categorical(df, cols, title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    for col in cols:
        if col not in df.columns:
            print(f"\n{col}: MISSING")
            continue

        print(f"\n{col}:")
        display(
            df[col]
            .astype("string")
            .fillna("<NA>")
            .value_counts(dropna=False)
            .reset_index()
            .rename(columns={"index": col, col: "count"})
            .head(50)
        )

def summarize_reconstruction(df, title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    diff_cols = [c for c in df.columns if c.endswith("_abs_diff_using_forecast_variance_candidate")]

    all_diff_cols = [c for c in df.columns if "_abs_diff_using_" in c]
    all_diff_cols += [c for c in df.columns if c.endswith("_candidate_vol_abs_diff")]

    for col in all_diff_cols:
        print(f"\n{col}:")
        display(
            df[col]
            .describe(percentiles=[0.5, 0.9, 0.99, 0.999])
            .to_frame()
        )
        print(f"Rows <= 1e-10: {(df[col] <= 1e-10).sum():,} / {len(df):,}")
        print(f"Rows >  1e-10: {(df[col] > 1e-10).sum():,} / {len(df):,}")

# ======================================================================================
# Basic panel summaries
# ======================================================================================

panel_summary("FORECAST — 07A unified_fds_no_min_return_oos_forecast_panel", forecast)
panel_summary("COMMON — 07A unified_common_row_panel", common)
panel_summary("DIAGNOSTIC — 07A unified_signal_threshold_diagnostic_panel", diagnostic)

# ======================================================================================
# Forecast duplicate diagnostics
# ======================================================================================

forecast_checked = add_reconstruction_checks(forecast, "forecast")
forecast_checked["_key_count"] = forecast_checked.groupby(["trade_date", "tenor"])["tenor"].transform("size")
forecast_checked["_is_duplicate_key"] = forecast_checked["_key_count"] > 1

dup = forecast_checked[forecast_checked["_is_duplicate_key"]].copy()
nondup = forecast_checked[~forecast_checked["_is_duplicate_key"]].copy()

print("\n" + "=" * 100)
print("FORECAST duplicate vs non-duplicate split")
print("=" * 100)
print(f"Forecast total rows:      {len(forecast_checked):,}")
print(f"Duplicate-key rows:       {len(dup):,}")
print(f"Non-duplicate-key rows:   {len(nondup):,}")
print(f"Duplicate unique keys:    {dup[['trade_date', 'tenor']].drop_duplicates().shape[0]:,}")
print(f"Non-duplicate unique keys:{nondup[['trade_date', 'tenor']].drop_duplicates().shape[0]:,}")

diagnostic_cols = [
    "model_spec",
    "model_source",
    "fit_status_candidate",
    "fit_status",
    "forecast_repair_scope",
    "is_back_frozen",
    "is_research_tenor",
    "tenor_bucket",
    "test_year",
    "test_year_candidate",
    "selected_alpha",
    "selected_alpha_candidate",
    "train_rows_used",
    "train_rows_used_candidate",
]

summarize_categorical(forecast_checked, diagnostic_cols, "FORECAST categorical diagnostics — all rows")
summarize_categorical(dup, diagnostic_cols, "FORECAST categorical diagnostics — duplicate-key rows only")
summarize_categorical(nondup, diagnostic_cols, "FORECAST categorical diagnostics — non-duplicate-key rows only")

summarize_reconstruction(forecast_checked, "FORECAST reconstruction diagnostics — all rows")
summarize_reconstruction(dup, "FORECAST reconstruction diagnostics — duplicate-key rows only")
summarize_reconstruction(nondup, "FORECAST reconstruction diagnostics — non-duplicate-key rows only")

# ======================================================================================
# Pair-level duplicate comparison
# ======================================================================================

print("\n" + "=" * 100)
print("Pair-level duplicate comparison")
print("=" * 100)

sort_cols = ["trade_date", "tenor"]

rank_cols = [
    "forecast_abs_diff_using_forecast_variance_candidate",
    "forecast_candidate_vol_abs_diff",
]

rank_col = "forecast_abs_diff_using_forecast_variance_candidate"

dup_sorted = dup.sort_values(sort_cols + [rank_col]).copy()
dup_sorted["_within_key_rank_by_candidate_vrp_diff"] = (
    dup_sorted.groupby(["trade_date", "tenor"]).cumcount() + 1
)

pair_summary = (
    dup_sorted
    .groupby("_within_key_rank_by_candidate_vrp_diff")
    .agg(
        rows=("tenor", "size"),
        min_date=("trade_date", "min"),
        max_date=("trade_date", "max"),
        avg_candidate_vrp_abs_diff=(rank_col, "mean"),
        median_candidate_vrp_abs_diff=(rank_col, "median"),
        max_candidate_vrp_abs_diff=(rank_col, "max"),
        avg_forecast_variance_candidate=("forecast_variance_candidate", "mean"),
        avg_candidate_forecast_vol_pct=("candidate_forecast_vol_pct", "mean"),
    )
    .reset_index()
)

pair_summary["min_date"] = pair_summary["min_date"].dt.date.astype(str)
pair_summary["max_date"] = pair_summary["max_date"].dt.date.astype(str)

display(pair_summary)

print("\nSample duplicated keys, ranked by candidate VRP reconstruction closeness:")
sample_cols = [
    "trade_date",
    "tenor",
    "_within_key_rank_by_candidate_vrp_diff",
    "model_spec",
    "model_source",
    "fit_status_candidate",
    "fit_status",
    "forecast_repair_scope",
    "is_back_frozen",
    "implied_variance",
    "forecast_variance_candidate",
    "forecast_variance_har_downside_v1",
    "baseline_forecast_variance",
    "model_vrp_log",
    "forecast_vrp_log_using_forecast_variance_candidate",
    "forecast_abs_diff_using_forecast_variance_candidate",
    "candidate_forecast_vol_pct",
    "forecast_candidate_vol_recalc",
    "forecast_candidate_vol_abs_diff",
    "predicted_log_variance_candidate",
    "candidate_log_rv_21d",
    "target_realized_variance",
]

sample_cols = [c for c in sample_cols if c in dup_sorted.columns]

display(
    dup_sorted[sample_cols]
    .sort_values(["trade_date", "tenor", "_within_key_rank_by_candidate_vrp_diff"])
    .head(100)
)

# ======================================================================================
# Check whether common/diagnostic panels are already deduped and internally consistent
# ======================================================================================

for name, df in [
    ("COMMON", common),
    ("DIAGNOSTIC", diagnostic),
]:
    checked = add_reconstruction_checks(df, name.lower())

    print("\n" + "=" * 100)
    print(f"{name} panel reconstruction and duplicate checks")
    print("=" * 100)

    dup_rows = int(checked.duplicated(["trade_date", "tenor"], keep=False).sum())
    print(f"Duplicate trade_date × tenor rows: {dup_rows:,}")

    if f"{name.lower()}_abs_diff_using_forecast_variance_candidate" in checked.columns:
        diff = checked[f"{name.lower()}_abs_diff_using_forecast_variance_candidate"]
        print(f"Rows where model_vrp_log matches candidate forecast variance <= 1e-10: {(diff <= 1e-10).sum():,} / {len(checked):,}")
        print(f"Rows where model_vrp_log differs candidate forecast variance >  1e-10: {(diff > 1e-10).sum():,} / {len(checked):,}")
        display(diff.describe(percentiles=[0.5, 0.9, 0.99, 0.999]).to_frame("abs_diff"))

    print("\nCategorical diagnostics:")
    for col in diagnostic_cols:
        if col in checked.columns:
            print(f"\n{name} — {col}:")
            display(
                checked[col]
                .astype("string")
                .fillna("<NA>")
                .value_counts(dropna=False)
                .reset_index()
                .rename(columns={"index": col, col: "count"})
                .head(20)
            )

# ======================================================================================
# Compare candidate deduped forecast rows to common and diagnostic panels
# ======================================================================================

print("\n" + "=" * 100)
print("Candidate dedupe methods")
print("=" * 100)

# Method A: choose row per date-tenor where model_vrp_log best matches log(implied / forecast_variance_candidate)
dedup_by_candidate_vrp = (
    forecast_checked
    .sort_values(["trade_date", "tenor", "forecast_abs_diff_using_forecast_variance_candidate"])
    .drop_duplicates(["trade_date", "tenor"], keep="first")
    .copy()
)

# Method B: choose row per date-tenor where candidate_forecast_vol_pct best matches sqrt(forecast_variance_candidate) * 100
if "forecast_candidate_vol_abs_diff" in forecast_checked.columns:
    dedup_by_candidate_vol = (
        forecast_checked
        .sort_values(["trade_date", "tenor", "forecast_candidate_vol_abs_diff"])
        .drop_duplicates(["trade_date", "tenor"], keep="first")
        .copy()
    )
else:
    dedup_by_candidate_vol = None

for name, df in [
    ("dedup_by_candidate_vrp", dedup_by_candidate_vrp),
    ("dedup_by_candidate_vol", dedup_by_candidate_vol),
]:
    if df is None:
        continue

    print("\n" + "-" * 100)
    print(name)
    print("-" * 100)
    print(f"Rows: {len(df):,}")
    print(f"Duplicate keys: {df.duplicated(['trade_date', 'tenor'], keep=False).sum():,}")
    print(f"Date range: {df['trade_date'].min().date()} to {df['trade_date'].max().date()}")
    print(f"Tenors: {sorted(df['tenor'].dropna().astype(int).unique().tolist())}")

    diff = df["forecast_abs_diff_using_forecast_variance_candidate"]
    print(f"VRP candidate reconstruction rows <= 1e-10: {(diff <= 1e-10).sum():,} / {len(df):,}")
    print(f"VRP candidate reconstruction rows >  1e-10: {(diff > 1e-10).sum():,} / {len(df):,}")
    display(diff.describe(percentiles=[0.5, 0.9, 0.99, 0.999]).to_frame("abs_diff"))

# ======================================================================================
# Save audit candidate for review
# ======================================================================================

audit_dir = PROJECT_ROOT / "data" / "audit" / "vrp_final_signal"
audit_dir.mkdir(parents=True, exist_ok=True)

audit_path = audit_dir / "07A_duplicate_block_diagnostic_preview.csv"

preview_cols = [
    "trade_date",
    "tenor",
    "_key_count",
    "_within_key_rank_by_candidate_vrp_diff",
    "model_spec",
    "model_source",
    "fit_status_candidate",
    "fit_status",
    "forecast_repair_scope",
    "is_back_frozen",
    "is_research_tenor",
    "implied_variance",
    "forecast_variance_candidate",
    "forecast_variance_har_downside_v1",
    "baseline_forecast_variance",
    "model_vrp_log",
    "forecast_vrp_log_using_forecast_variance_candidate",
    "forecast_abs_diff_using_forecast_variance_candidate",
    "candidate_forecast_vol_pct",
    "forecast_candidate_vol_recalc",
    "forecast_candidate_vol_abs_diff",
    "predicted_log_variance_candidate",
    "candidate_log_rv_21d",
    "target_realized_variance",
]

preview_cols = [c for c in preview_cols if c in dup_sorted.columns]

dup_sorted[preview_cols].head(5000).to_csv(audit_path, index=False)

print("\n" + "=" * 100)
print("Saved duplicate diagnostic preview")
print("=" * 100)
print(audit_path)

07A duplicate-block diagnostic — source existence
forecast_panel       exists=True  path=C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\07A_unified_fds_no_min_return_oos_forecast_panel_20200102_20260701_20260704_203242.parquet
common_row_panel     exists=True  path=C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\07A_unified_common_row_panel_20200102_20260701_20260704_203242.parquet
diagnostic_panel     exists=True  path=C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\07A_unified_signal_threshold_diagnostic_panel_20200102_20260701_20260704_203242.parquet

FORECAST — 07A unified_fds_no_min_return_oos_forecast_panel
Rows: 31,140
Cols: 89
Date range: 2019-01-02 to 2026-07-01
Unique dates: 1,884
Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
Unique trade_date × tenor keys: 16,452
Duplicate trade_date × tenor rows: 29,376

Tenor-count distribution:


,unique_tenors_per_date,date_count
0,7,252
1,9,1632



COMMON — 07A unified_common_row_panel
Rows: 29,376
Cols: 93
Date range: 2020-01-02 to 2026-07-01
Unique dates: 1,632
Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
Unique trade_date × tenor keys: 14,688
Duplicate trade_date × tenor rows: 29,376

Tenor-count distribution:


,unique_tenors_per_date,date_count
0,9,1632



DIAGNOSTIC — 07A unified_signal_threshold_diagnostic_panel
Rows: 29,376
Cols: 99
Date range: 2020-01-02 to 2026-07-01
Unique dates: 1,632
Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
Unique trade_date × tenor keys: 14,688
Duplicate trade_date × tenor rows: 29,376

Tenor-count distribution:


...


Candidate dedupe methods

----------------------------------------------------------------------------------------------------
dedup_by_candidate_vrp
----------------------------------------------------------------------------------------------------
Rows: 16,452
Duplicate keys: 0
Date range: 2019-01-02 to 2026-07-01
Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
VRP candidate reconstruction rows <= 1e-10: 16,452 / 16,452
VRP candidate reconstruction rows >  1e-10: 0 / 16,452


,abs_diff
count,16452.0
mean,0.0
std,0.0
min,0.0
50%,0.0
90%,0.0
99%,0.0
99.9%,0.0
max,0.0



----------------------------------------------------------------------------------------------------
dedup_by_candidate_vol
----------------------------------------------------------------------------------------------------
Rows: 16,452
Duplicate keys: 0
Date range: 2019-01-02 to 2026-07-01
Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
VRP candidate reconstruction rows <= 1e-10: 16,452 / 16,452
VRP candidate reconstruction rows >  1e-10: 0 / 16,452


,abs_diff
count,16452.0
mean,0.0
std,0.0
min,0.0
50%,0.0
90%,0.0
99%,0.0
99.9%,0.0
max,0.0



Saved duplicate diagnostic preview
C:\Users\patri\vrp_project\data\audit\vrp_final_signal\07A_duplicate_block_diagnostic_preview.csv


In [5]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)

# ======================================================================================
# Scope
# ======================================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

FORECAST_PANEL_PATH = Path(
    r"C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1"
    r"\07A_unified_fds_no_min_return_oos_forecast_panel_20200102_20260701_20260704_203242.parquet"
)

SIGNAL_RULES_PATH = Path(
    r"C:\Users\patri\vrp_project\data\processed\vrp_core_secondary_tertiary_independent_sizing_research_v1"
    r"\09S_core_secondary_consistent_secondary_locked_rules_long_20260705_221829.parquet"
)

SIZING_RULES_PATH = Path(
    r"C:\Users\patri\vrp_project\data\processed\vrp_core_secondary_tertiary_independent_sizing_research_v1"
    r"\15S_final_one_trade_per_day_naked_put_sizing_lock_rules_20260706_142800.parquet"
)

SELECTION_RULE_PATH = Path(
    r"C:\Users\patri\vrp_project\data\audit\vrp_core_secondary_tertiary_independent_sizing_research_v1"
    r"\15S_final_one_trade_per_day_naked_put_selection_rule_20260706_142800.json"
)

RV21D_PATH = Path(
    r"C:\Users\patri\vrp_project\data\processed\market_data\spy_realized_vol_history_v1.parquet"
)

LOCKED_16S_SELECTED_PATH = Path(
    r"C:\Users\patri\vrp_project\data\processed\vrp_core_secondary_tertiary_independent_sizing_research_v1"
    r"\16S_locked_cumulative_pnl_timeseries_20260706_143802.parquet"
)

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "vrp_final_signal"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

Z_3M_WINDOW = 63
Z_1Y_WINDOW = 252
Z_DDOF = 1

EXPECTED_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

print("=" * 100)
print("VRP final signal logic dry run")
print("=" * 100)
print("No production files will be written.")
print(f"Run timestamp: {RUN_TIMESTAMP}")

# ======================================================================================
# Source existence
# ======================================================================================

paths = {
    "forecast_panel": FORECAST_PANEL_PATH,
    "signal_rules": SIGNAL_RULES_PATH,
    "sizing_rules": SIZING_RULES_PATH,
    "selection_rule": SELECTION_RULE_PATH,
    "rv21d": RV21D_PATH,
    "locked_16s_selected_optional": LOCKED_16S_SELECTED_PATH,
}

print("\n" + "=" * 100)
print("Source file existence")
print("=" * 100)

for name, path in paths.items():
    print(f"{name:32s} exists={path.exists()}  path={path}")

required_missing = [
    name for name, path in paths.items()
    if name != "locked_16s_selected_optional" and not path.exists()
]

if required_missing:
    raise FileNotFoundError(f"Missing required files: {required_missing}")

# ======================================================================================
# Load sources
# ======================================================================================

forecast_raw = pd.read_parquet(FORECAST_PANEL_PATH)
signal_rules_raw = pd.read_parquet(SIGNAL_RULES_PATH)
sizing_rules_raw = pd.read_parquet(SIZING_RULES_PATH)
rv21d_raw = pd.read_parquet(RV21D_PATH)

with open(SELECTION_RULE_PATH, "r", encoding="utf-8") as f:
    selection_rule_json = json.load(f)

locked_16s = None
if LOCKED_16S_SELECTED_PATH.exists():
    locked_16s = pd.read_parquet(LOCKED_16S_SELECTED_PATH)

# Normalize source dtypes
forecast_raw["trade_date"] = pd.to_datetime(forecast_raw["trade_date"], errors="coerce")
forecast_raw["tenor"] = pd.to_numeric(forecast_raw["tenor"], errors="coerce").astype("Int64")

rv21d_raw["trade_date"] = pd.to_datetime(rv21d_raw["trade_date"], errors="coerce")

signal_rules_raw["tenor"] = pd.to_numeric(signal_rules_raw["tenor"], errors="coerce").astype("Int64")
sizing_rules_raw["tenor"] = pd.to_numeric(sizing_rules_raw["tenor"], errors="coerce").astype("Int64")

# ======================================================================================
# Filter to unified FDS candidate rows
# ======================================================================================

base = forecast_raw[
    (forecast_raw["model_spec"].astype(str) == "unified_fds_no_min_return")
    & (forecast_raw["model_source"].astype(str) == "unified_fds_no_min_return_oos_refit")
    & (forecast_raw["fit_status_candidate"].astype(str) == "candidate_fit")
].copy()

base = base.sort_values(["trade_date", "tenor"]).reset_index(drop=True)

print("\n" + "=" * 100)
print("Unified FDS candidate base")
print("=" * 100)
print(f"Rows: {len(base):,}")
print(f"Date range: {base['trade_date'].min().date()} to {base['trade_date'].max().date()}")
print(f"Unique dates: {base['trade_date'].nunique():,}")
print(f"Tenors: {sorted(base['tenor'].dropna().astype(int).unique().tolist())}")
print(f"Duplicate trade_date × tenor rows: {base.duplicated(['trade_date', 'tenor'], keep=False).sum():,}")

# ======================================================================================
# Recompute final denominator fields
# ======================================================================================

base["implied_variance_final"] = base["implied_variance"].astype(float)
base["vix_style_vol_final"] = base["vix_style_vol"].astype(float)
base["forecast_variance_final"] = base["forecast_variance_candidate"].astype(float)
base["forecast_vol_final"] = np.sqrt(base["forecast_variance_final"]) * 100.0
base["model_vrp_log_final"] = np.log(base["implied_variance_final"] / base["forecast_variance_final"])
base["rsi14_final"] = base["RSI14"].astype(float)

base["source_model_vrp_log"] = base["model_vrp_log"].astype(float)
base["source_model_vrp_z_3m"] = base["model_vrp_z_3m"].astype(float)
base["source_model_vrp_z_1y"] = base["model_vrp_z_1y"].astype(float)

base["source_vs_final_model_vrp_log_diff"] = (
    base["source_model_vrp_log"] - base["model_vrp_log_final"]
)

# ======================================================================================
# Recompute prior-only z-scores from unified final VRP log
# ======================================================================================

base = base.sort_values(["tenor", "trade_date"]).copy()

def prior_only_z(series: pd.Series, window: int, ddof: int = 1) -> pd.Series:
    prior = series.shift(1)
    mean = prior.rolling(window=window, min_periods=window).mean()
    std = prior.rolling(window=window, min_periods=window).std(ddof=ddof)
    return (series - mean) / std

base["z_3m_final"] = (
    base.groupby("tenor", group_keys=False)["model_vrp_log_final"]
    .apply(lambda s: prior_only_z(s, Z_3M_WINDOW, Z_DDOF))
)

base["z_1y_final"] = (
    base.groupby("tenor", group_keys=False)["model_vrp_log_final"]
    .apply(lambda s: prior_only_z(s, Z_1Y_WINDOW, Z_DDOF))
)

base = base.sort_values(["trade_date", "tenor"]).reset_index(drop=True)

print("\n" + "=" * 100)
print("Final recomputed field checks")
print("=" * 100)

print("Source model_vrp_log vs recomputed model_vrp_log_final absolute diff:")
display(
    base["source_vs_final_model_vrp_log_diff"]
    .abs()
    .describe(percentiles=[0.5, 0.9, 0.99, 0.999])
    .to_frame("abs_diff")
)

print("\nZ-score non-null counts by tenor:")
z_summary = (
    base.groupby("tenor")
    .agg(
        rows=("trade_date", "size"),
        z_3m_non_null=("z_3m_final", lambda s: int(s.notna().sum())),
        z_1y_non_null=("z_1y_final", lambda s: int(s.notna().sum())),
        first_date=("trade_date", "min"),
        last_date=("trade_date", "max"),
    )
    .reset_index()
)
z_summary["first_date"] = z_summary["first_date"].dt.date.astype(str)
z_summary["last_date"] = z_summary["last_date"].dt.date.astype(str)
display(z_summary)

# ======================================================================================
# Join canonical RV21D
# ======================================================================================

rv21d = rv21d_raw[["trade_date", "spy_close", "rv21d_vol_pct"]].copy()
rv21d["rv21d_vol_pct_final"] = rv21d["rv21d_vol_pct"].astype(float)

base = base.merge(
    rv21d[["trade_date", "spy_close", "rv21d_vol_pct_final"]],
    on="trade_date",
    how="left",
    validate="many_to_one",
)

print("\n" + "=" * 100)
print("RV21D join")
print("=" * 100)
print(f"Rows after RV21D join: {len(base):,}")
print(f"Missing rv21d_vol_pct_final: {base['rv21d_vol_pct_final'].isna().sum():,}")
print(f"Latest date: {base['trade_date'].max().date()}")
display(
    base.sort_values(["trade_date", "tenor"])
    .tail(18)[["trade_date", "tenor", "spy_close", "rv21d_vol_pct_final"]]
)

# ======================================================================================
# Prepare active signal and sizing rules
# ======================================================================================

signal_rules = signal_rules_raw.copy()

if signal_rules["include_tenor"].dtype != bool:
    signal_rules["include_tenor"] = signal_rules["include_tenor"].astype(str).str.lower().map(
        {"true": True, "1": True, "yes": True, "false": False, "0": False, "no": False}
    ).fillna(False)

active_rules = signal_rules[
    (signal_rules["include_tenor"] == True)
    & (signal_rules["signal_layer"].isin(["Core", "Secondary"]))
].copy()

sizing_rules = sizing_rules_raw.copy()

if "is_locked" in sizing_rules.columns:
    if sizing_rules["is_locked"].dtype != bool:
        sizing_rules["is_locked"] = sizing_rules["is_locked"].astype(str).str.lower().map(
            {"true": True, "1": True, "yes": True, "false": False, "0": False, "no": False}
        ).fillna(False)

if "is_final_sizing_lock" in sizing_rules.columns:
    if sizing_rules["is_final_sizing_lock"].dtype != bool:
        sizing_rules["is_final_sizing_lock"] = sizing_rules["is_final_sizing_lock"].astype(str).str.lower().map(
            {"true": True, "1": True, "yes": True, "false": False, "0": False, "no": False}
        ).fillna(False)

active_sizing = sizing_rules[
    (sizing_rules["signal_layer"].isin(["Core", "Secondary"]))
    & (sizing_rules.get("is_locked", True) == True)
    & (sizing_rules.get("is_final_sizing_lock", True) == True)
].copy()

print("\n" + "=" * 100)
print("Active rules and sizing")
print("=" * 100)

print(f"Active signal rule rows: {len(active_rules):,}")
display(
    active_rules[
        [
            "signal_layer",
            "signal_priority",
            "tenor_bucket",
            "tenor",
            "model_vrp_log_min",
            "model_vrp_z_3m_min",
            "model_vrp_z_1y_min",
            "RSI14_max",
            "rv21d_vol_pct_min",
        ]
    ].sort_values(["signal_priority", "tenor"])
)

print(f"\nActive sizing rows: {len(active_sizing):,}")
display(
    active_sizing[
        [
            "sleeve_id",
            "signal_layer",
            "tenor_bucket",
            "tenor",
            "locked_size_pct",
            "sizing_quality_score",
            "win_rate",
            "pnl_per_day_expected_loss_1pct_positive",
            "signal_layer_order",
            "tenor_bucket_order",
            "display_order",
        ]
    ].sort_values(["signal_layer_order", "tenor"])
)

# ======================================================================================
# Apply signal thresholds
# ======================================================================================

candidate_base_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "implied_variance_final",
    "vix_style_vol_final",
    "forecast_variance_final",
    "forecast_vol_final",
    "model_vrp_log_final",
    "z_3m_final",
    "z_1y_final",
    "rsi14_final",
    "rv21d_vol_pct_final",
    "spy_close",
    "model_spec",
    "model_source",
    "fit_status_candidate",
    "forecast_repair_scope",
]

candidate_base_cols = [c for c in candidate_base_cols if c in base.columns]

candidates = base[candidate_base_cols].merge(
    active_rules[
        [
            "signal_layer",
            "signal_priority",
            "tenor",
            "tenor_bucket",
            "model_vrp_log_min",
            "model_vrp_z_3m_min",
            "model_vrp_z_1y_min",
            "RSI14_max",
            "rv21d_vol_pct_min",
            "comparison_operator_model_vrp_log",
            "comparison_operator_model_vrp_z_3m",
            "comparison_operator_model_vrp_z_1y",
            "comparison_operator_RSI14",
            "comparison_operator_rv21d_vol_pct",
            "stack_lock_version",
            "stack_lock_decision_id",
            "lock_version",
            "lock_decision_id",
        ]
    ],
    on=["tenor", "tenor_bucket"],
    how="inner",
    validate="many_to_many",
)

supported_ops = {
    "comparison_operator_model_vrp_log": [">"],
    "comparison_operator_model_vrp_z_3m": [">"],
    "comparison_operator_model_vrp_z_1y": [">"],
    "comparison_operator_RSI14": ["<"],
    "comparison_operator_rv21d_vol_pct": [">"],
}

for op_col, allowed in supported_ops.items():
    actual = sorted(candidates[op_col].dropna().astype(str).unique().tolist())
    if actual != allowed:
        raise RuntimeError(f"Unexpected operators in {op_col}: {actual}. Expected {allowed}")

candidates["pass_model_vrp_log"] = candidates["model_vrp_log_final"] > candidates["model_vrp_log_min"]
candidates["pass_z_3m"] = candidates["z_3m_final"] > candidates["model_vrp_z_3m_min"]
candidates["pass_z_1y"] = candidates["z_1y_final"] > candidates["model_vrp_z_1y_min"]
candidates["pass_rsi14"] = candidates["rsi14_final"] < candidates["RSI14_max"]
candidates["pass_rv21d"] = candidates["rv21d_vol_pct_final"] > candidates["rv21d_vol_pct_min"]

pass_cols = [
    "pass_model_vrp_log",
    "pass_z_3m",
    "pass_z_1y",
    "pass_rsi14",
    "pass_rv21d",
]

candidates["qualified"] = candidates[pass_cols].all(axis=1)

# Join sizing to every candidate
sizing_join_cols = [
    "sleeve_id",
    "signal_layer",
    "tenor_bucket",
    "tenor",
    "locked_size_pct",
    "locked_size_label",
    "sizing_quality_score",
    "win_rate",
    "pnl_per_day_expected_loss_1pct_positive",
    "signal_layer_order",
    "tenor_bucket_order",
    "display_order",
    "sizing_lock_version",
    "sizing_lock_decision_id",
    "program_type",
    "pricing_basis_label",
    "not_defined_spread_sizing",
]

sizing_join_cols = [c for c in sizing_join_cols if c in active_sizing.columns]

candidates = candidates.merge(
    active_sizing[sizing_join_cols],
    on=["signal_layer", "tenor_bucket", "tenor"],
    how="left",
    validate="many_to_one",
)

missing_size_all = candidates["locked_size_pct"].isna().sum()
missing_size_qualified = candidates.loc[candidates["qualified"], "locked_size_pct"].isna().sum()

if missing_size_qualified > 0:
    raise RuntimeError(f"Qualified candidates missing sizing rows: {missing_size_qualified:,}")

print("\n" + "=" * 100)
print("Signal qualification counts")
print("=" * 100)

print(f"Candidate rows: {len(candidates):,}")
print(f"Qualified candidate rows: {candidates['qualified'].sum():,}")
print(f"Missing locked_size_pct among all candidates: {missing_size_all:,}")
print(f"Missing locked_size_pct among qualified candidates: {missing_size_qualified:,}")

pass_dist = (
    candidates[candidates["qualified"]]
    .groupby(["signal_layer", "tenor_bucket", "tenor"])
    .size()
    .reset_index(name="qualified_rows")
    .sort_values(["signal_layer", "tenor"])
)

display(pass_dist)

# ======================================================================================
# Apply 15S one-trade-per-day selection rule
# ======================================================================================

qualified = candidates[candidates["qualified"]].copy()

qualified = qualified.sort_values(
    [
        "trade_date",
        "locked_size_pct",
        "signal_layer_order",
        "sizing_quality_score",
        "win_rate",
        "pnl_per_day_expected_loss_1pct_positive",
        "tenor",
        "display_order",
    ],
    ascending=[
        True,
        False,  # highest locked size
        True,   # Core before Secondary if size tied
        False,  # higher sizing quality
        False,  # higher win rate
        True,   # lower positive 1% expected loss
        False,  # longer tenor
        True,
    ],
).reset_index(drop=True)

qualified["_selection_rank"] = qualified.groupby("trade_date").cumcount() + 1
selected = qualified[qualified["_selection_rank"] == 1].copy()

print("\n" + "=" * 100)
print("Selected trades — dry run")
print("=" * 100)

print(f"Selected rows: {len(selected):,}")
if len(selected) > 0:
    print(f"Selected date range: {selected['trade_date'].min().date()} to {selected['trade_date'].max().date()}")
    print(f"Max selected rows per date: {selected.groupby('trade_date').size().max()}")

selected_dist = (
    selected.groupby(["signal_layer", "tenor_bucket", "tenor", "sleeve_id", "locked_size_pct"])
    .size()
    .reset_index(name="selected_count")
    .sort_values(["signal_layer", "tenor"])
)

display(selected_dist)

selected_by_year = (
    selected.assign(year=selected["trade_date"].dt.year)
    .groupby(["year", "signal_layer", "tenor"])
    .size()
    .reset_index(name="selected_count")
    .sort_values(["year", "signal_layer", "tenor"])
)

print("\nSelected trades by year/layer/tenor:")
display(selected_by_year)

# ======================================================================================
# Build dry-run panel view
# ======================================================================================

panel = base.copy()

core_pass = (
    candidates[(candidates["signal_layer"] == "Core") & (candidates["qualified"])]
    [["trade_date", "tenor"]]
    .drop_duplicates()
    .assign(core_pass=True)
)

secondary_pass = (
    candidates[(candidates["signal_layer"] == "Secondary") & (candidates["qualified"])]
    [["trade_date", "tenor"]]
    .drop_duplicates()
    .assign(secondary_pass=True)
)

panel = panel.merge(core_pass, on=["trade_date", "tenor"], how="left")
panel = panel.merge(secondary_pass, on=["trade_date", "tenor"], how="left")

panel["core_pass"] = panel["core_pass"].fillna(False).astype(bool)
panel["secondary_pass"] = panel["secondary_pass"].fillna(False).astype(bool)

selected_marker = selected[
    [
        "trade_date",
        "tenor",
        "signal_layer",
        "tenor_bucket",
        "sleeve_id",
        "locked_size_pct",
        "locked_size_label",
        "sizing_quality_score",
        "win_rate",
        "pnl_per_day_expected_loss_1pct_positive",
    ]
].copy()

selected_marker = selected_marker.rename(
    columns={
        "signal_layer": "selected_layer",
        "tenor_bucket": "selected_tenor_bucket",
        "sleeve_id": "selected_sleeve_id",
        "locked_size_pct": "selected_size_pct",
        "locked_size_label": "selected_size_label",
        "sizing_quality_score": "selected_sizing_quality_score",
        "win_rate": "selected_win_rate",
        "pnl_per_day_expected_loss_1pct_positive": "selected_1pct_expected_loss_positive",
    }
)

panel = panel.merge(
    selected_marker,
    on=["trade_date", "tenor"],
    how="left",
    validate="one_to_one",
)

panel["selected"] = panel["selected_layer"].notna()
panel["selected_tenor"] = np.where(panel["selected"], panel["tenor"], np.nan)

final_panel_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "model_spec",
    "model_source",
    "fit_status_candidate",
    "forecast_repair_scope",
    "implied_variance_final",
    "vix_style_vol_final",
    "forecast_variance_final",
    "forecast_vol_final",
    "model_vrp_log_final",
    "z_3m_final",
    "z_1y_final",
    "rsi14_final",
    "rv21d_vol_pct_final",
    "spy_close",
    "source_model_vrp_log",
    "source_model_vrp_z_3m",
    "source_model_vrp_z_1y",
    "source_vs_final_model_vrp_log_diff",
    "core_pass",
    "secondary_pass",
    "selected",
    "selected_layer",
    "selected_tenor_bucket",
    "selected_tenor",
    "selected_sleeve_id",
    "selected_size_pct",
    "selected_size_label",
    "selected_sizing_quality_score",
    "selected_win_rate",
    "selected_1pct_expected_loss_positive",
]

final_panel_cols = [c for c in final_panel_cols if c in panel.columns]
panel_out = panel[final_panel_cols].copy()

# ======================================================================================
# Compare to locked 16S selected artifact, if available
# ======================================================================================

comparison_summary = {}
comparison_detail = pd.DataFrame()

print("\n" + "=" * 100)
print("Comparison to locked 16S selected artifact")
print("=" * 100)

if locked_16s is None:
    print("16S artifact not found; skipping comparison.")
else:
    locked = locked_16s.copy()
    if "trade_date" in locked.columns:
        locked["trade_date"] = pd.to_datetime(locked["trade_date"], errors="coerce")
    
    print(f"16S rows: {len(locked):,}")
    print("16S columns:")
    for c in locked.columns:
        print(f"  {c}")

    # Column detection
    locked_layer_col = None
    for c in ["signal_layer", "selected_layer", "layer"]:
        if c in locked.columns:
            locked_layer_col = c
            break

    locked_tenor_col = None
    for c in ["tenor", "selected_tenor", "dte"]:
        if c in locked.columns:
            locked_tenor_col = c
            break

    if locked_layer_col is None or locked_tenor_col is None:
        print("Could not detect locked 16S layer/tenor columns. Skipping exact match comparison.")
    else:
        selected_compare = selected[
            ["trade_date", "signal_layer", "tenor", "sleeve_id", "locked_size_pct"]
        ].rename(
            columns={
                "signal_layer": "dry_signal_layer",
                "tenor": "dry_tenor",
                "sleeve_id": "dry_sleeve_id",
                "locked_size_pct": "dry_locked_size_pct",
            }
        )

        locked_compare_cols = ["trade_date", locked_layer_col, locked_tenor_col]
        if "sleeve_id" in locked.columns:
            locked_compare_cols.append("sleeve_id")
        if "locked_size_pct" in locked.columns:
            locked_compare_cols.append("locked_size_pct")

        locked_compare = locked[locked_compare_cols].copy()
        locked_compare = locked_compare.rename(
            columns={
                locked_layer_col: "locked_signal_layer",
                locked_tenor_col: "locked_tenor",
                "sleeve_id": "locked_sleeve_id",
                "locked_size_pct": "locked_size_pct",
            }
        )

        locked_compare["locked_tenor"] = pd.to_numeric(locked_compare["locked_tenor"], errors="coerce")
        selected_compare["dry_tenor"] = pd.to_numeric(selected_compare["dry_tenor"], errors="coerce")

        comparison_detail = locked_compare.merge(
            selected_compare,
            on="trade_date",
            how="outer",
            indicator=True,
        )

        comparison_detail["layer_match"] = (
            comparison_detail["locked_signal_layer"].astype("string")
            == comparison_detail["dry_signal_layer"].astype("string")
        )

        comparison_detail["tenor_match"] = (
            comparison_detail["locked_tenor"].astype("float")
            == comparison_detail["dry_tenor"].astype("float")
        )

        comparison_detail["exact_match"] = (
            (comparison_detail["_merge"] == "both")
            & comparison_detail["layer_match"]
            & comparison_detail["tenor_match"]
        )

        comparison_summary = {
            "locked_rows": int(len(locked_compare)),
            "dry_selected_rows": int(len(selected_compare)),
            "both_dates": int((comparison_detail["_merge"] == "both").sum()),
            "locked_only_dates": int((comparison_detail["_merge"] == "left_only").sum()),
            "dry_only_dates": int((comparison_detail["_merge"] == "right_only").sum()),
            "exact_layer_tenor_matches": int(comparison_detail["exact_match"].sum()),
            "common_date_mismatches": int(
                ((comparison_detail["_merge"] == "both") & (~comparison_detail["exact_match"])).sum()
            ),
        }

        print(json.dumps(comparison_summary, indent=2))

        print("\nLocked 16S distribution:")
        display(
            locked_compare
            .groupby(["locked_signal_layer", "locked_tenor"])
            .size()
            .reset_index(name="locked_count")
            .sort_values(["locked_signal_layer", "locked_tenor"])
        )

        print("\nDry-run selected distribution:")
        display(
            selected_compare
            .groupby(["dry_signal_layer", "dry_tenor"])
            .size()
            .reset_index(name="dry_count")
            .sort_values(["dry_signal_layer", "dry_tenor"])
        )

        print("\nSample comparison mismatches:")
        display(
            comparison_detail[
                (comparison_detail["_merge"] != "both") | (~comparison_detail["exact_match"])
            ]
            .sort_values("trade_date")
            .head(100)
        )

# ======================================================================================
# Validation
# ======================================================================================

validation_rows = []

def add_check(check, status, detail):
    validation_rows.append(
        {
            "check": check,
            "status": "PASS" if bool(status) else "FAIL",
            "detail": detail,
        }
    )

add_check(
    "filtered_to_unified_fds_candidate_rows",
    len(base) > 0
    and set(base["model_spec"].astype(str).unique()) == {"unified_fds_no_min_return"}
    and set(base["model_source"].astype(str).unique()) == {"unified_fds_no_min_return_oos_refit"}
    and set(base["fit_status_candidate"].astype(str).unique()) == {"candidate_fit"},
    f"rows={len(base):,}",
)

add_check(
    "one_row_per_trade_date_tenor",
    panel_out.duplicated(["trade_date", "tenor"], keep=False).sum() == 0,
    f"duplicate_rows={panel_out.duplicated(['trade_date', 'tenor'], keep=False).sum():,}",
)

actual_tenors = sorted(panel_out["tenor"].dropna().astype(int).unique().tolist())
add_check(
    "expected_tenor_grid",
    actual_tenors == EXPECTED_TENORS,
    f"actual_tenors={actual_tenors}",
)

add_check(
    "positive_implied_and_forecast_variance",
    (panel_out["implied_variance_final"] > 0).all()
    and (panel_out["forecast_variance_final"] > 0).all(),
    "implied_variance_final and forecast_variance_final must be positive",
)

forecast_vol_diff = (
    panel_out["forecast_vol_final"]
    - np.sqrt(panel_out["forecast_variance_final"]) * 100
).abs()

add_check(
    "forecast_vol_reconstruction",
    (forecast_vol_diff <= 1e-10).all(),
    f"max_abs_diff={forecast_vol_diff.max()}",
)

vrp_log_diff = (
    panel_out["model_vrp_log_final"]
    - np.log(panel_out["implied_variance_final"] / panel_out["forecast_variance_final"])
).abs()

add_check(
    "model_vrp_log_final_reconstruction",
    (vrp_log_diff <= 1e-10).all(),
    f"max_abs_diff={vrp_log_diff.max()}",
)

add_check(
    "rv21d_join_complete",
    panel_out["rv21d_vol_pct_final"].notna().all(),
    f"missing={panel_out['rv21d_vol_pct_final'].isna().sum():,}",
)

add_check(
    "active_signal_rule_count",
    len(active_rules) == 13,
    f"active_rules={len(active_rules):,}",
)

add_check(
    "active_sizing_rule_count",
    len(active_sizing) == 13,
    f"active_sizing={len(active_sizing):,}",
)

add_check(
    "qualified_candidates_have_sizing",
    missing_size_qualified == 0,
    f"missing_size_qualified={missing_size_qualified:,}",
)

max_selected_per_date = selected.groupby("trade_date").size().max() if len(selected) > 0 else 0

add_check(
    "max_one_selected_trade_per_day",
    max_selected_per_date <= 1,
    f"max_selected_per_date={max_selected_per_date}",
)

validation = pd.DataFrame(validation_rows)

print("\n" + "=" * 100)
print("Dry-run validation")
print("=" * 100)
display(validation)

if (validation["status"] == "FAIL").any():
    print("Validation failures found. Do not proceed to production script.")
else:
    print("All dry-run hard validations passed.")

# ======================================================================================
# Save audit outputs
# ======================================================================================

panel_audit_path = AUDIT_DIR / f"vrp_final_signal_logic_dry_run_panel_{RUN_TIMESTAMP}.csv"
selected_audit_path = AUDIT_DIR / f"vrp_final_signal_logic_dry_run_selected_{RUN_TIMESTAMP}.csv"
candidates_audit_path = AUDIT_DIR / f"vrp_final_signal_logic_dry_run_candidates_{RUN_TIMESTAMP}.csv"
validation_audit_path = AUDIT_DIR / f"vrp_final_signal_logic_dry_run_validation_{RUN_TIMESTAMP}.csv"
selected_dist_path = AUDIT_DIR / f"vrp_final_signal_logic_dry_run_selected_distribution_{RUN_TIMESTAMP}.csv"
comparison_path = AUDIT_DIR / f"vrp_final_signal_logic_dry_run_16S_comparison_{RUN_TIMESTAMP}.csv"
manifest_path = AUDIT_DIR / f"vrp_final_signal_logic_dry_run_manifest_{RUN_TIMESTAMP}.json"

panel_out.to_csv(panel_audit_path, index=False)
selected.to_csv(selected_audit_path, index=False)
candidates.to_csv(candidates_audit_path, index=False)
validation.to_csv(validation_audit_path, index=False)
selected_dist.to_csv(selected_dist_path, index=False)

if not comparison_detail.empty:
    comparison_detail.to_csv(comparison_path, index=False)

manifest = {
    "run_timestamp": RUN_TIMESTAMP,
    "run_type": "final_signal_logic_dry_run_only",
    "no_production_outputs_written": True,
    "forecast_panel_path": str(FORECAST_PANEL_PATH),
    "signal_rules_path": str(SIGNAL_RULES_PATH),
    "sizing_rules_path": str(SIZING_RULES_PATH),
    "selection_rule_path": str(SELECTION_RULE_PATH),
    "rv21d_path": str(RV21D_PATH),
    "locked_16s_selected_path": str(LOCKED_16S_SELECTED_PATH) if LOCKED_16S_SELECTED_PATH.exists() else None,
    "z_3m_window": Z_3M_WINDOW,
    "z_1y_window": Z_1Y_WINDOW,
    "z_ddof": Z_DDOF,
    "base_rows": int(len(base)),
    "panel_rows": int(len(panel_out)),
    "candidate_rows": int(len(candidates)),
    "qualified_candidate_rows": int(candidates["qualified"].sum()),
    "selected_rows": int(len(selected)),
    "selected_min_date": str(selected["trade_date"].min().date()) if len(selected) > 0 else None,
    "selected_max_date": str(selected["trade_date"].max().date()) if len(selected) > 0 else None,
    "comparison_summary": comparison_summary,
    "validation": validation.to_dict(orient="records"),
    "audit_outputs": {
        "panel": str(panel_audit_path),
        "selected": str(selected_audit_path),
        "candidates": str(candidates_audit_path),
        "validation": str(validation_audit_path),
        "selected_distribution": str(selected_dist_path),
        "comparison": str(comparison_path) if not comparison_detail.empty else None,
    },
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 100)
print("Saved dry-run audit outputs")
print("=" * 100)
print(panel_audit_path)
print(selected_audit_path)
print(candidates_audit_path)
print(validation_audit_path)
print(selected_dist_path)
if not comparison_detail.empty:
    print(comparison_path)
print(manifest_path)

print("\nDONE — final signal logic dry run completed.")

VRP final signal logic dry run
No production files will be written.
Run timestamp: 20260706_222901

Source file existence
forecast_panel                   exists=True  path=C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\07A_unified_fds_no_min_return_oos_forecast_panel_20200102_20260701_20260704_203242.parquet
signal_rules                     exists=True  path=C:\Users\patri\vrp_project\data\processed\vrp_core_secondary_tertiary_independent_sizing_research_v1\09S_core_secondary_consistent_secondary_locked_rules_long_20260705_221829.parquet
sizing_rules                     exists=True  path=C:\Users\patri\vrp_project\data\processed\vrp_core_secondary_tertiary_independent_sizing_research_v1\15S_final_one_trade_per_day_naked_put_sizing_lock_rules_20260706_142800.parquet
selection_rule                   exists=True  path=C:\Users\patri\vrp_project\data\audit\vrp_core_secondary_tertiary_independent_sizing_research_v1\15S_final_one_trade_per_day_naked_put_

,abs_diff
count,14688.000000
mean,0.174492
std,0.211922
min,0.000002
50%,0.124514
90%,0.338130
99%,1.199144
99.9%,2.193120
max,2.596017



Z-score non-null counts by tenor:


,tenor,rows,z_3m_non_null,z_1y_non_null,first_date,last_date
0,9,1632,1569,1380,2020-01-02,2026-07-01
1,12,1632,1569,1380,2020-01-02,2026-07-01
2,15,1632,1569,1380,2020-01-02,2026-07-01
3,18,1632,1569,1380,2020-01-02,2026-07-01
4,21,1632,1569,1380,2020-01-02,2026-07-01
5,24,1632,1569,1380,2020-01-02,2026-07-01
6,27,1632,1569,1380,2020-01-02,2026-07-01
7,30,1632,1569,1380,2020-01-02,2026-07-01
8,33,1632,1569,1380,2020-01-02,2026-07-01



RV21D join
Rows after RV21D join: 14,688
Missing rv21d_vol_pct_final: 0
Latest date: 2026-07-01


...


Sample comparison mismatches:


,trade_date,locked_signal_layer,locked_tenor,locked_sleeve_id,locked_size_pct,dry_signal_layer,dry_tenor,dry_sleeve_id,dry_locked_size_pct,_merge,layer_match,tenor_match,exact_match
9,2021-09-10,Secondary,18.0,Secondary_Front_18D,0.0275,Secondary,15,Secondary_Front_15D,0.020,both,True,False,False
156,2023-01-17,Secondary,18.0,Secondary_Front_18D,0.0275,Secondary,15,Secondary_Front_15D,0.020,both,True,False,False
158,2023-01-26,Secondary,18.0,Secondary_Front_18D,0.0275,Secondary,15,Secondary_Front_15D,0.020,both,True,False,False
159,2023-01-27,Secondary,18.0,Secondary_Front_18D,0.0275,Secondary,15,Secondary_Front_15D,0.020,both,True,False,False
172,2023-03-16,Secondary,18.0,Secondary_Front_18D,0.0275,Secondary,15,Secondary_Front_15D,0.020,both,True,False,False
175,2023-03-21,Secondary,18.0,Secondary_Front_18D,0.0275,Secondary,15,Secondary_Front_15D,0.020,both,True,False,False
177,2023-03-23,Secondary,18.0,Secondary_Front_18D,0.0275,Secondary,12,Secondary_Front_12D,0.015,both,True,False,False
183,2023-04-18,Secondary,18.0,Secondary_Front_18D,0.0275,Secondary,12,Secondary_Front_12D,0.015,both,True,False,False
187,2023-05-24,Secondary,18.0,Secondary_Front_18D,0.0275,Secondary,12,Secondary_Front_12D,0.015,both,True,False,False
195,2023-10-17,Secondary,18.0,Secondary_Front_18D,0.0275,Secondary,15,Secondary_Front_15D,0.020,both,True,False,False



Dry-run validation


,check,status,detail
0,filtered_to_unified_fds_candidate_rows,PASS,"rows=14,688"
1,one_row_per_trade_date_tenor,PASS,duplicate_rows=0
2,expected_tenor_grid,PASS,"actual_tenors=[9, 12, 15, 18, 21, 24, 27, 30, 33]"
3,positive_implied_and_forecast_variance,PASS,implied_variance_final and forecast_variance_final must be positive
4,forecast_vol_reconstruction,PASS,max_abs_diff=0.0
5,model_vrp_log_final_reconstruction,PASS,max_abs_diff=0.0
6,rv21d_join_complete,PASS,missing=0
7,active_signal_rule_count,PASS,active_rules=13
8,active_sizing_rule_count,PASS,active_sizing=13
9,qualified_candidates_have_sizing,PASS,missing_size_qualified=0


All dry-run hard validations passed.

Saved dry-run audit outputs
C:\Users\patri\vrp_project\data\audit\vrp_final_signal\vrp_final_signal_logic_dry_run_panel_20260706_222901.csv
C:\Users\patri\vrp_project\data\audit\vrp_final_signal\vrp_final_signal_logic_dry_run_selected_20260706_222901.csv
C:\Users\patri\vrp_project\data\audit\vrp_final_signal\vrp_final_signal_logic_dry_run_candidates_20260706_222901.csv
C:\Users\patri\vrp_project\data\audit\vrp_final_signal\vrp_final_signal_logic_dry_run_validation_20260706_222901.csv
C:\Users\patri\vrp_project\data\audit\vrp_final_signal\vrp_final_signal_logic_dry_run_selected_distribution_20260706_222901.csv
C:\Users\patri\vrp_project\data\audit\vrp_final_signal\vrp_final_signal_logic_dry_run_16S_comparison_20260706_222901.csv
C:\Users\patri\vrp_project\data\audit\vrp_final_signal\vrp_final_signal_logic_dry_run_manifest_20260706_222901.json

DONE — final signal logic dry run completed.
